In [ ]:
import pandas as pd
import io

In [ ]:
def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
    sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
    sandbox_log =  sections[0].strip()
    activities_log = sections[1].split('Trade History:')[0]
    # sandbox_log_list = [json.loads(line) for line in sandbox_log.split('\n')]
    trade_history =  json.loads(sections[1].split('Trade History:')[1])
    # sandbox_log_df = pd.DataFrame(sandbox_log_list)
    market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    trade_history_df = pd.json_normalize(trade_history)
    return market_data_df, trade_history_df

In [ ]:
market_data_df, _ = _process_data_('./round-4-island-data-bottle/round_3_results.log')

In [ ]:
market_data_df

In [ ]:
df = market_data_df.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

In [ ]:
df

In [ ]:
def get_prev_returns(df, col, its):
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_from_{its}_its_ago"] = (df[col] - df[prev_col]) / df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    return df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

def get_centered_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_centered_with_{its}_its"] = (df[future_col] - df[prev_col])/df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    df.drop(columns=[future_col], inplace=True)
    return df

In [ ]:
df.columns

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500]
symbols = ['CHOCOLATE', 'GIFT_BASKET', 'ORCHIDS', 'ROSES', 'STARFRUIT', 'STRAWBERRIES']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")

    # Add lagged and future returns columns for each symbol
    for symbol in symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)

    for target_symbol in symbols:
        print(f"Target Symbol: {target_symbol}")

        # Get the feature columns (lagged returns from other symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and not col.startswith(target_symbol)]

        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"

        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()

        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]

        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)

        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)

        # Make predictions on the training data
        y_pred = model.predict(X)

        # Calculate the R-squared
        r2 = r2_score(y, y_pred)

        # Print the R-squared in bold and large font if it is greater than 0.01
        if r2 > 0.01:
            print(f"\033[1m\033[4mR-squared: {r2:.4f}\033[0m")
        else:
            print(f"R-squared: {r2:.4f}")

        print()

    df_copy = df_copy[['timestamp'] + symbols]
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000]
predictor_symbols = ['STARFRUIT']
responder_symbols = ['ROSES']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in predictor_symbols + responder_symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in responder_symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from predictor symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + predictor_symbols + responder_symbols]
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000]
predictor_symbols = ["GIFT_BASKET"]
responder_symbols = ['ORCHIDS']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in predictor_symbols + responder_symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in responder_symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from predictor symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + predictor_symbols + responder_symbols]
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000]
predictor_symbols = ['STARFRUIT', "GIFT_BASKET"]
responder_symbols = ['ORCHIDS']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in predictor_symbols + responder_symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in responder_symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from predictor symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + predictor_symbols + responder_symbols]
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000]
predictor_symbols = ['STARFRUIT']
responder_symbols = ['ROSES','CHOCOLATE','STRAWBERRIES',"GIFT_BASKET"]

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in predictor_symbols + responder_symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in responder_symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from predictor symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        
        print()
    
    df_copy = df_copy[['timestamp'] + predictor_symbols + responder_symbols]
    print()

# backtest

```
Timeframe: 400 iterations
Target Symbol: ROSES
Learned Equation:
ROSES_returns_in_400_its = 1.3429 * STARFRUIT_returns_from_400_its_ago + 0.6912 * STRAWBERRIES_returns_from_400_its_ago
R-squared: 0.2091
p-value: 0.0000
```

```
Timeframe: 400 iterations
Target Symbol: ROSES
Learned Equation:
ROSES_returns_in_400_its = 1.2158 * STARFRUIT_returns_from_400_its_ago
R-squared: 0.1006
p-value: 0.0000
```


```
Target Symbol: STRAWBERRIES
Learned Equation:
STRAWBERRIES_returns_in_800_its = 0.4900 * STARFRUIT_returns_from_800_its_ago
R-squared: 0.3189
```

```
Target Symbol: GIFT_BASKET
Learned Equation:
GIFT_BASKET_returns_in_900_its = 0.3303 * STARFRUIT_returns_from_900_its_ago
R-squared: 0.1215
```

In [ ]:
def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
    sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
    sandbox_log =  sections[0].strip()
    activities_log = sections[1].split('Trade History:')[0]
    # sandbox_log_list = [json.loads(line) for line in sandbox_log.split('\n')]
    trade_history =  json.loads(sections[1].split('Trade History:')[1])
    # sandbox_log_df = pd.DataFrame(sandbox_log_list)
    market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    trade_history_df = pd.json_normalize(trade_history)
    return market_data_df, trade_history_df

In [ ]:
market_data_df, _ = _process_data_('./round-4-island-data-bottle/round_3_results.log')

In [ ]:
market_data_df

In [ ]:
df = market_data_df.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

In [ ]:
df

In [ ]:
def get_prev_returns(df, col, its):
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_from_{its}_its_ago"] = (df[col] - df[prev_col]) / df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    return df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

def get_centered_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_centered_with_{its}_its"] = (df[future_col] - df[prev_col])/df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    df.drop(columns=[future_col], inplace=True)
    return df

In [ ]:
df.columns

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500]
symbols = ['CHOCOLATE', 'GIFT_BASKET', 'ORCHIDS', 'ROSES', 'STARFRUIT', 'STRAWBERRIES']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")

    # Add lagged and future returns columns for each symbol
    for symbol in symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)

    for target_symbol in symbols:
        print(f"Target Symbol: {target_symbol}")

        # Get the feature columns (lagged returns from other symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and not col.startswith(target_symbol)]

        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"

        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()

        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]

        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)

        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)

        # Make predictions on the training data
        y_pred = model.predict(X)

        # Calculate the R-squared
        r2 = r2_score(y, y_pred)

        # Print the R-squared in bold and large font if it is greater than 0.01
        if r2 > 0.01:
            print(f"\033[1m\033[4mR-squared: {r2:.4f}\033[0m")
        else:
            print(f"R-squared: {r2:.4f}")

        print()

    df_copy = df_copy[['timestamp'] + symbols]
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000]
predictor_symbols = ['STARFRUIT']
responder_symbols = ['ROSES']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in predictor_symbols + responder_symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in responder_symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from predictor symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + predictor_symbols + responder_symbols]
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000]
predictor_symbols = ["GIFT_BASKET"]
responder_symbols = ['ORCHIDS']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in predictor_symbols + responder_symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in responder_symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from predictor symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + predictor_symbols + responder_symbols]
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000]
predictor_symbols = ['STARFRUIT', "GIFT_BASKET"]
responder_symbols = ['ORCHIDS']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in predictor_symbols + responder_symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in responder_symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from predictor symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + predictor_symbols + responder_symbols]
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000]
predictor_symbols = ['STARFRUIT']
responder_symbols = ['ROSES','CHOCOLATE','STRAWBERRIES',"GIFT_BASKET"]

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in predictor_symbols + responder_symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in responder_symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from predictor symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        
        print()
    
    df_copy = df_copy[['timestamp'] + predictor_symbols + responder_symbols]
    print()

# backtest

```
Timeframe: 400 iterations
Target Symbol: ROSES
Learned Equation:
ROSES_returns_in_400_its = 1.3429 * STARFRUIT_returns_from_400_its_ago + 0.6912 * STRAWBERRIES_returns_from_400_its_ago
R-squared: 0.2091
p-value: 0.0000
```

```
Timeframe: 400 iterations
Target Symbol: ROSES
Learned Equation:
ROSES_returns_in_400_its = 1.2158 * STARFRUIT_returns_from_400_its_ago
R-squared: 0.1006
p-value: 0.0000
```


```
Target Symbol: STRAWBERRIES
Learned Equation:
STRAWBERRIES_returns_in_800_its = 0.4900 * STARFRUIT_returns_from_800_its_ago
R-squared: 0.3189
```

```
Target Symbol: GIFT_BASKET
Learned Equation:
GIFT_BASKET_returns_in_900_its = 0.3303 * STARFRUIT_returns_from_900_its_ago
R-squared: 0.1215
```

In [ ]:
market_data_df.columns

In [ ]:
import numpy as np

def market_maker_mid(row):
    if row['product'] == 'STARFRUIT':
        bid_prices = [row['bid_price_1'], row['bid_price_2'], row['bid_price_3']]
        bid_volumes = [row['bid_volume_1'], row['bid_volume_2'], row['bid_volume_3']]
        ask_prices = [row['ask_price_1'], row['ask_price_2'], row['ask_price_3']]
        ask_volumes = [row['ask_volume_1'], row['ask_volume_2'], row['ask_volume_3']]

        best_bid = np.nan
        best_ask = np.nan

        for bid_price, bid_volume in zip(bid_prices, bid_volumes):
            if bid_volume > 15:
                best_bid = bid_price
                break

        for ask_price, ask_volume in zip(ask_prices, ask_volumes):
            if ask_volume > 15:
                best_ask = ask_price
                break

        if not np.isnan(best_bid) and not np.isnan(best_ask):
            return (best_bid + best_ask) / 2
        else:
            return np.nan
    else:
        return row['mid_price']

market_data_df['mid_price'] = market_data_df.apply(market_maker_mid, axis=1)

In [ ]:
market_data_df

In [ ]:
df_backtest = df.copy()[['timestamp', 'STARFRUIT', 'ROSES', 'STRAWBERRIES', 'GIFT_BASKET']]

df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 500)
df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 800)
df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 900)
df_backtest = get_future_returns(df_backtest, 'ROSES', 500)
df_backtest = get_future_returns(df_backtest, 'STRAWBERRIES', 800)
df_backtest = get_future_returns(df_backtest, 'GIFT_BASKET', 900)



STARFRUIT_BETA = 1.255
STRAWBERRIES_BETA = 0.49
GIFT_BASKET_BETA = 0.33


df_backtest['ROSES_pred_returns'] = STARFRUIT_BETA * df_backtest['STARFRUIT_returns_from_500_its_ago'] 
df_backtest['STRAWBERRIES_pred_returns'] = STRAWBERRIES_BETA * df_backtest['STARFRUIT_returns_from_800_its_ago'] 
df_backtest['GIFT_BASKET_pred_returns'] = GIFT_BASKET_BETA * df_backtest['STARFRUIT_returns_from_900_its_ago'] 


display(df_backtest['ROSES_pred_returns'].describe())
display(df_backtest['STRAWBERRIES_pred_returns'].describe())
display(df_backtest['GIFT_BASKET_pred_returns'].describe())

In [ ]:
ROSES_take_threshold = 0.004
ROSES_clear_threshold = 0.0005

STRAWBERRIES_take_threshold = 0.00125
STRAWBERRIES_clear_threshold = 0.0002

GIFT_BASKET_take_threshold = 0.002
GIFT_BASKET_clear_threshold = 0.0002


# Function to generate signals based on predicted returns and thresholds
def generate_signals(row, symbol):
    pred_returns = row[f'{symbol}_pred_returns']
    take_threshold = eval(f'{symbol}_take_threshold')
    clear_threshold = eval(f'{symbol}_clear_threshold')
    
    if pred_returns <= -take_threshold:
        return 'SHORT'
    elif -clear_threshold < pred_returns < clear_threshold:
        return 'CLEAR'
    elif pred_returns >= take_threshold:
        return 'LONG'
    else:
        return None

# Generate signals for each symbol
symbols = ['ROSES', 'STRAWBERRIES', 'GIFT_BASKET']
for symbol in symbols:
    df_backtest[f'{symbol}_signal'] = df_backtest.apply(lambda row: generate_signals(row, symbol), axis=1)
    df_backtest[f'last_{symbol}_signal'] = df_backtest[f'{symbol}_signal'].fillna(method='ffill')

# Set the position size for each symbol
positions = {'ROSES': 60, "STRAWBERRIES": 350, "GIFT_BASKET": 60}

# Calculate target positions for each symbol
for symbol in symbols:
    df_backtest.loc[df_backtest[f'last_{symbol}_signal'] == 'CLEAR', f'{symbol}_target_position'] = 0
    df_backtest.loc[df_backtest[f'last_{symbol}_signal'] == 'SHORT', f'{symbol}_target_position'] = -positions[symbol]
    df_backtest.loc[df_backtest[f'last_{symbol}_signal'] == 'LONG', f'{symbol}_target_position'] = positions[symbol]
    df_backtest[f'{symbol}_target_position'] = df_backtest[f'{symbol}_target_position'].fillna(0)

# Calculate target position changes for each symbol
for symbol in symbols:
    df_backtest[f'{symbol}_target_position_change'] = df_backtest[f'{symbol}_target_position'].diff(1)

# Calculate cash, cumulative cash, position mark, and PNL for each symbol
for symbol in symbols:
    df_backtest[f'{symbol}_cash'] = -df_backtest[f'{symbol}_target_position_change'] * df_backtest[symbol]
    df_backtest[f'{symbol}_cumulative_cash'] = df_backtest[f'{symbol}_cash'].cumsum()
    df_backtest[f'{symbol}_position_mark'] = df_backtest[f'{symbol}_target_position'] * df_backtest[symbol]
    df_backtest[f'{symbol}_pnl'] = df_backtest[f'{symbol}_position_mark'] + df_backtest[f'{symbol}_cumulative_cash']

# Display the PNL for each symbol
for symbol in symbols:
    print(f"{symbol} PNL:")
    display(df_backtest[f'{symbol}_pnl'].tail(3))

In [ ]:
df_backtest

In [ ]:
import plotly.graph_objects as go

# Create traces for ROSES
trace_roses_pred_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_pred_returns'],
    name='ROSES Predicted Returns',
    yaxis='y1'
)

trace_roses_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_returns_in_500_its'],
    name='ROSES Returns',
    yaxis='y3'
)

trace_roses_price = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES'],
    name='ROSES Price',
    yaxis='y4'
)

trace_roses_pnl = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_pnl'],
    name='ROSES PnL',
    yaxis='y2'
)

# Create traces for STRAWBERRIES
trace_strawberries_pred_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES_pred_returns'],
    name='STRAWBERRIES Predicted Returns',
    yaxis='y5'
)

trace_strawberries_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES_returns_in_800_its'],
    name='STRAWBERRIES Returns',
    yaxis='y7'
)

trace_strawberries_price = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES'],
    name='STRAWBERRIES Price',
    yaxis='y8'
)

trace_strawberries_pnl = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES_pnl'],
    name='STRAWBERRIES PnL',
    yaxis='y10'
)

# Create traces for GIFT_BASKET
trace_gift_basket_pred_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['GIFT_BASKET_pred_returns'],
    name='GIFT_BASKET Predicted Returns',
    yaxis='y9'
)

trace_gift_basket_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['GIFT_BASKET_returns_in_900_its'],
    name='GIFT_BASKET Returns',
    yaxis='y11'
)

trace_gift_basket_price = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['GIFT_BASKET'],
    name='GIFT_BASKET Price',
    yaxis='y12'
)

trace_gift_basket_pnl = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['GIFT_BASKET_pnl'],
    name='GIFT_BASKET PnL',
    yaxis='y13'
)

# Create trace for STARFRUIT
trace_starfruit = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STARFRUIT'],
    name='STARFRUIT',
    yaxis='y6'
)

# Create the layout for the graph
layout = go.Layout(
    title='Multiple Symbols - Predicted Returns, Returns, Prices, and PnL',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(title='ROSES Predicted Returns', side='left'),
    yaxis2=dict(title='ROSES PnL', side='right', overlaying='y'),
    yaxis3=dict(title='ROSES Returns', side='right', overlaying='y', position=0.95),
    yaxis4=dict(title='ROSES Price', side='right', overlaying='y', position=0.95),
    yaxis5=dict(title='STRAWBERRIES Predicted Returns', side='right', overlaying='y', position=0.95),
    yaxis6=dict(title='STARFRUIT', side='right', overlaying='y', position=0.95),
    yaxis7=dict(title='STRAWBERRIES Returns', side='right', overlaying='y', position=0.95),
    yaxis8=dict(title='STRAWBERRIES Price', side='right', overlaying='y', position=0.95),
    yaxis9=dict(title='GIFT_BASKET Predicted Returns', side='right', overlaying='y', position=0.95),
    yaxis10=dict(title='STRAWBERRIES PnL', side='right', overlaying='y', position=0.95),
    yaxis11=dict(title='GIFT_BASKET Returns', side='right', overlaying='y', position=0.95),
    yaxis12=dict(title='GIFT_BASKET Price', side='right', overlaying='y', position=0.95),
    yaxis13=dict(title='GIFT_BASKET PnL', side='right', overlaying='y', position=0.95)
)

# Create the figure and add the traces and layout
fig = go.Figure(data=[
    trace_roses_pred_returns, trace_roses_returns, trace_roses_price, trace_roses_pnl,
    trace_strawberries_pred_returns, trace_strawberries_returns, trace_strawberries_price, trace_strawberries_pnl,
    trace_gift_basket_pred_returns, trace_gift_basket_returns, trace_gift_basket_price, trace_gift_basket_pnl,
    trace_starfruit
], layout=layout)

# Show the plot
fig.show()

# backtest day 0 (out of distribution)

In [ ]:
file_name = f"../round1/round-{1}-island-data-bottle/prices_round_{1}_day_0.csv"
df_round_1 = pd.read_csv(file_name, sep=';')

file_name = f"../round3/round-{3}-island-data-bottle/prices_round_{3}_day_0.csv"
df_round_3 = pd.read_csv(file_name, sep=';')

import numpy as np

def market_maker_mid(row):
    if row['product'] == 'STARFRUIT':
        bid_prices = [row['bid_price_1'], row['bid_price_2'], row['bid_price_3']]
        bid_volumes = [row['bid_volume_1'], row['bid_volume_2'], row['bid_volume_3']]
        ask_prices = [row['ask_price_1'], row['ask_price_2'], row['ask_price_3']]
        ask_volumes = [row['ask_volume_1'], row['ask_volume_2'], row['ask_volume_3']]

        best_bid = np.nan
        best_ask = np.nan

        for bid_price, bid_volume in zip(bid_prices, bid_volumes):
            if bid_volume > 15:
                best_bid = bid_price
                break

        for ask_price, ask_volume in zip(ask_prices, ask_volumes):
            if ask_volume > 15:
                best_ask = ask_price
                break

        if not np.isnan(best_bid) and not np.isnan(best_ask):
            return (best_bid + best_ask) / 2
        else:
            return np.nan
    else:
        return row['mid_price']

df_round_1['mid_price'] = df_round_1.apply(market_maker_mid, axis=1)

df_round_1 = df_round_1.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

df_round_3 = df_round_3.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

df = df_round_1.merge(df_round_3, on='timestamp', how='inner')

In [ ]:
df

In [ ]:
df_backtest = df.copy()[['timestamp', 'STARFRUIT', 'ROSES', 'STRAWBERRIES']]

df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 500)
df_backtest = get_prev_returns(df_backtest, 'STRAWBERRIES', 500)
df_backtest = get_future_returns(df_backtest, 'ROSES', 500)



STARFRUIT_BETA = 1.25

df_backtest['ROSES_pred_returns_in_500_its'] = STARFRUIT_BETA * df_backtest['STARFRUIT_returns_from_500_its_ago']

take_threshold = 0.004
clear_threshold = 0.0005


df_backtest['roses_signal'] = df_backtest.apply(lambda row: "SHORT" if row['ROSES_pred_returns_in_500_its'] <= -take_threshold else "CLEAR" if (row['ROSES_pred_returns_in_500_its'] > -clear_threshold and row['ROSES_pred_returns_in_500_its'] < clear_threshold) else "LONG" if row['ROSES_pred_returns_in_500_its'] >= take_threshold else None,  axis=1)

df_backtest['last_roses_signal'] = df_backtest['roses_signal'].fillna(method='ffill')

df_backtest

POSITION = 60

df_backtest.loc[df_backtest['last_roses_signal'] == 'CLEAR', 'roses_target_position'] = 0
df_backtest.loc[df_backtest['last_roses_signal'] == 'SHORT', 'roses_target_position'] = -POSITION
df_backtest.loc[df_backtest['last_roses_signal'] == 'LONG', 'roses_target_position'] = POSITION

df_backtest

df_backtest['roses_target_position'] = df_backtest['roses_target_position'].fillna(0)

df_backtest

df_backtest['roses_target_position_change'] = df_backtest['roses_target_position'].diff(1)

df_backtest = df_backtest.dropna()

df_backtest['cash'] = -df_backtest['roses_target_position_change'] * df_backtest['ROSES']

df_backtest['cumulative_cash'] = df_backtest['cash'].cumsum()

df_backtest['position_mark'] = df_backtest['roses_target_position'] * df_backtest['ROSES']

df_backtest['pnl'] = df_backtest['position_mark'] + df_backtest['cumulative_cash']

df_backtest['pnl'].tail(3)

In [ ]:
df_backtest

In [ ]:
import plotly.graph_objects as go

# Create the first trace for ROSES_pred_returns_in_500_its
trace1 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_pred_returns_in_500_its'],
    name='ROSES Predicted Returns',
    yaxis='y1'
)

# Create the second trace for pnl
trace2 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['pnl'],
    name='PnL',
    yaxis='y2'
)

# Create the third trace for ROSES
trace3 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_returns_in_500_its'],
    name='ROSES_returns',
    yaxis='y3'
)

trace4 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES'],
    name='ROSES',
    yaxis='y4'
)

trace5 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STARFRUIT'],
    name='STARFRUIT',
    yaxis='y5'
)

trace6 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES'],
    name='STRAWBERRIES',
    yaxis='y6'
)



# Create the layout for the graph
layout = go.Layout(
    title='ROSES, Predicted Returns, and PnL',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(
        title='ROSES Predicted Returns',
        side='left'
    ),
    yaxis2=dict(
        title='PnL',
        side='right',
        overlaying='y'
    ),
    yaxis3=dict(
        title='ROSES_returns',
        side='right',
        overlaying='y',
        position=0.95
    ),
    yaxis4=dict(
        title='ROSES',
        side='right',
        overlaying='y',
        position=0.92
    ),
    yaxis5=dict(
        title='STARFRUIT',
        side='right',
        overlaying='y',
        position=0.9
    ),
    yaxis6=dict(
        title='STRAWBERRIES',
        side='right',
        overlaying='y',
        position=0.9
    )
)

# Create the figure and add the traces and layout
fig = go.Figure(data=[trace1, trace2, trace3, trace4, trace5,trace6], layout=layout)

# Show the plot
fig.show()

In [ ]:
df_backtest.to_csv('backtest_roses.csv')

# generalized

In [ ]:
file_name = f"../round1/round-{1}-island-data-bottle/prices_round_{1}_day_0.csv"
df_round_1 = pd.read_csv(file_name, sep=';')

file_name = f"../round3/round-{3}-island-data-bottle/prices_round_{3}_day_0.csv"
df_round_3 = pd.read_csv(file_name, sep=';')

import numpy as np

def market_maker_mid(row):
    if row['product'] == 'STARFRUIT':
        bid_prices = [row['bid_price_1'], row['bid_price_2'], row['bid_price_3']]
        bid_volumes = [row['bid_volume_1'], row['bid_volume_2'], row['bid_volume_3']]
        ask_prices = [row['ask_price_1'], row['ask_price_2'], row['ask_price_3']]
        ask_volumes = [row['ask_volume_1'], row['ask_volume_2'], row['ask_volume_3']]

        best_bid = np.nan
        best_ask = np.nan

        for bid_price, bid_volume in zip(bid_prices, bid_volumes):
            if bid_volume > 15:
                best_bid = bid_price
                break

        for ask_price, ask_volume in zip(ask_prices, ask_volumes):
            if ask_volume > 15:
                best_ask = ask_price
                break

        if not np.isnan(best_bid) and not np.isnan(best_ask):
            return (best_bid + best_ask) / 2
        else:
            return np.nan
    else:
        return row['mid_price']

df_round_1['mid_price'] = df_round_1.apply(market_maker_mid, axis=1)

df_round_1 = df_round_1.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

df_round_3 = df_round_3.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

df = df_round_1.merge(df_round_3, on='timestamp', how='inner')

In [ ]:
def backtest_strategy(df, responder, responder_timeframe, predictors, predictor_betas, predictor_timeframes, take_threshold, clear_threshold, positions):
    df_backtest = df.copy()[[responder] + predictors]
    
    # Calculate predictor returns
    for i, predictor in enumerate(predictors):
        df_backtest = get_prev_returns(df_backtest, predictor, predictor_timeframes[i])
    
    # Calculate responder future returns
    df_backtest = get_future_returns(df_backtest, responder, responder_timeframe)
    
    # Calculate predicted returns for the responder
    df_backtest[f'{responder}_pred_returns'] = 0
    for i, predictor in enumerate(predictors):
        df_backtest[f'{responder}_pred_returns'] += predictor_betas[i] * df_backtest[f'{predictor}_returns_from_{predictor_timeframes[i]}_its_ago']
    
    # Generate signals based on predicted returns and thresholds
    def generate_signals(row):
        pred_returns = row[f'{responder}_pred_returns']
        if pred_returns <= -take_threshold:
            return 'SHORT'
        elif -clear_threshold < pred_returns < clear_threshold:
            return 'CLEAR'
        elif pred_returns >= take_threshold:
            return 'LONG'
        else:
            return None
    
    df_backtest[f'{responder}_signal'] = df_backtest.apply(generate_signals, axis=1)
    df_backtest[f'last_{responder}_signal'] = df_backtest[f'{responder}_signal'].fillna(method='ffill')
    
    # Calculate target positions
    df_backtest.loc[df_backtest[f'last_{responder}_signal'] == 'CLEAR', f'{responder}_target_position'] = 0
    df_backtest.loc[df_backtest[f'last_{responder}_signal'] == 'SHORT', f'{responder}_target_position'] = -positions
    df_backtest.loc[df_backtest[f'last_{responder}_signal'] == 'LONG', f'{responder}_target_position'] = positions
    df_backtest[f'{responder}_target_position'] = df_backtest[f'{responder}_target_position'].fillna(0)
    
    # Calculate target position changes
    df_backtest[f'{responder}_target_position_change'] = df_backtest[f'{responder}_target_position'].diff(1)
    
    # Calculate cash, cumulative cash, position mark, and PNL
    df_backtest[f'{responder}_cash'] = -df_backtest[f'{responder}_target_position_change'] * df_backtest[responder]
    df_backtest[f'{responder}_cumulative_cash'] = df_backtest[f'{responder}_cash'].cumsum()
    df_backtest[f'{responder}_position_mark'] = df_backtest[f'{responder}_target_position'] * df_backtest[responder]
    df_backtest[f'{responder}_pnl'] = df_backtest[f'{responder}_position_mark'] + df_backtest[f'{responder}_cumulative_cash']
    
    return df_backtest

In [ ]:
responder = 'STRAWBERRIES'
responder_timeframe = 800
predictors = [ 'STARFRUIT']
predictor_betas = [0.49]
predictor_timeframes = [500]
take_threshold = 0.00125
clear_threshold = 0.0002
positions = 350

df_backtest = backtest_strategy(df, responder, responder_timeframe, predictors, predictor_betas, predictor_timeframes, take_threshold, clear_threshold, positions)

print(f"{responder} PNL:")
display(df_backtest[f'{responder}_pnl'].tail(3))

In [ ]:
market_data_df.columns

In [ ]:
import numpy as np

def market_maker_mid(row):
    if row['product'] == 'STARFRUIT':
        bid_prices = [row['bid_price_1'], row['bid_price_2'], row['bid_price_3']]
        bid_volumes = [row['bid_volume_1'], row['bid_volume_2'], row['bid_volume_3']]
        ask_prices = [row['ask_price_1'], row['ask_price_2'], row['ask_price_3']]
        ask_volumes = [row['ask_volume_1'], row['ask_volume_2'], row['ask_volume_3']]

        best_bid = np.nan
        best_ask = np.nan

        for bid_price, bid_volume in zip(bid_prices, bid_volumes):
            if bid_volume > 15:
                best_bid = bid_price
                break

        for ask_price, ask_volume in zip(ask_prices, ask_volumes):
            if ask_volume > 15:
                best_ask = ask_price
                break

        if not np.isnan(best_bid) and not np.isnan(best_ask):
            return (best_bid + best_ask) / 2
        else:
            return np.nan
    else:
        return row['mid_price']

market_data_df['mid_price'] = market_data_df.apply(market_maker_mid, axis=1)

In [ ]:
market_data_df

In [ ]:
df_backtest = df.copy()[['timestamp', 'STARFRUIT', 'ROSES', 'STRAWBERRIES', 'GIFT_BASKET']]

df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 500)
df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 800)
df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 900)
df_backtest = get_future_returns(df_backtest, 'ROSES', 500)
df_backtest = get_future_returns(df_backtest, 'STRAWBERRIES', 800)
df_backtest = get_future_returns(df_backtest, 'GIFT_BASKET', 900)



STARFRUIT_BETA = 1.255
STRAWBERRIES_BETA = 0.49
GIFT_BASKET_BETA = 0.33


df_backtest['ROSES_pred_returns'] = STARFRUIT_BETA * df_backtest['STARFRUIT_returns_from_500_its_ago'] 
df_backtest['STRAWBERRIES_pred_returns'] = STRAWBERRIES_BETA * df_backtest['STARFRUIT_returns_from_800_its_ago'] 
df_backtest['GIFT_BASKET_pred_returns'] = GIFT_BASKET_BETA * df_backtest['STARFRUIT_returns_from_900_its_ago'] 


display(df_backtest['ROSES_pred_returns'].describe())
display(df_backtest['STRAWBERRIES_pred_returns'].describe())
display(df_backtest['GIFT_BASKET_pred_returns'].describe())

In [ ]:
df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 500)
df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 800)
df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 900)
df_backtest = get_prev_returns(df_backtest, 'STRAWBERRIES', 500)
df_backtest = get_future_returns(df_backtest, 'ROSES', 500)

In [ ]:
STARFRUIT_BETA = 1.255
STRAWBERRIES_BETA = 0

In [ ]:
df_backtest['ROSES_pred_returns_in_500_its'] = STARFRUIT_BETA * df_backtest['STARFRUIT_returns_from_500_its_ago'] + STRAWBERRIES_BETA * df_backtest['STRAWBERRIES_returns_from_500_its_ago']

In [ ]:
df_backtest

In [ ]:
df_backtest['ROSES_pred_returns_in_500_its'].describe()

In [ ]:
take_threshold = 0.004
clear_threshold = 0.0005

In [ ]:

df_backtest['roses_signal'] = df_backtest.apply(lambda row: "SHORT" if row['ROSES_pred_returns_in_500_its'] <= -take_threshold else "CLEAR" if (row['ROSES_pred_returns_in_500_its'] > -clear_threshold and row['ROSES_pred_returns_in_500_its'] < clear_threshold) else "LONG" if row['ROSES_pred_returns_in_500_its'] >= take_threshold else None,  axis=1)

In [ ]:
df_backtest['last_roses_signal'] = df_backtest['roses_signal'].fillna(method='ffill')

In [ ]:
df_backtest

In [ ]:
POSITION = 60

In [ ]:
df_backtest.loc[df_backtest['last_roses_signal'] == 'CLEAR', 'roses_target_position'] = 0
df_backtest.loc[df_backtest['last_roses_signal'] == 'SHORT', 'roses_target_position'] = -POSITION
df_backtest.loc[df_backtest['last_roses_signal'] == 'LONG', 'roses_target_position'] = POSITION

In [ ]:
df_backtest

In [ ]:
df_backtest['roses_target_position'] = df_backtest['roses_target_position'].fillna(0)

In [ ]:
df_backtest

In [ ]:
df_backtest['roses_target_position_change'] = df_backtest['roses_target_position'].diff(1)

In [ ]:
df_backtest = df_backtest.dropna()

In [ ]:
df_backtest['cash'] = -df_backtest['roses_target_position_change'] * df_backtest['ROSES']

df_backtest['cumulative_cash'] = df_backtest['cash'].cumsum()

df_backtest['position_mark'] = df_backtest['roses_target_position'] * df_backtest['ROSES']

df_backtest['pnl'] = df_backtest['position_mark'] + df_backtest['cumulative_cash']

df_backtest['pnl'].tail(3)

In [ ]:
ROSES_take_threshold = 0.004
ROSES_clear_threshold = 0.0005

STRAWBERRIES_take_threshold = 0.00125
STRAWBERRIES_clear_threshold = 0.0002

GIFT_BASKET_take_threshold = 0.002
GIFT_BASKET_clear_threshold = 0.0002


# Function to generate signals based on predicted returns and thresholds
def generate_signals(row, symbol):
    pred_returns = row[f'{symbol}_pred_returns']
    take_threshold = eval(f'{symbol}_take_threshold')
    clear_threshold = eval(f'{symbol}_clear_threshold')
    
    if pred_returns <= -take_threshold:
        return 'SHORT'
    elif -clear_threshold < pred_returns < clear_threshold:
        return 'CLEAR'
    elif pred_returns >= take_threshold:
        return 'LONG'
    else:
        return None

# Generate signals for each symbol
symbols = ['ROSES', 'STRAWBERRIES', 'GIFT_BASKET']
for symbol in symbols:
    df_backtest[f'{symbol}_signal'] = df_backtest.apply(lambda row: generate_signals(row, symbol), axis=1)
    df_backtest[f'last_{symbol}_signal'] = df_backtest[f'{symbol}_signal'].fillna(method='ffill')

# Set the position size for each symbol
positions = {'ROSES': 60, "STRAWBERRIES": 350, "GIFT_BASKET": 60}

# Calculate target positions for each symbol
for symbol in symbols:
    df_backtest.loc[df_backtest[f'last_{symbol}_signal'] == 'CLEAR', f'{symbol}_target_position'] = 0
    df_backtest.loc[df_backtest[f'last_{symbol}_signal'] == 'SHORT', f'{symbol}_target_position'] = -positions[symbol]
    df_backtest.loc[df_backtest[f'last_{symbol}_signal'] == 'LONG', f'{symbol}_target_position'] = positions[symbol]
    df_backtest[f'{symbol}_target_position'] = df_backtest[f'{symbol}_target_position'].fillna(0)

# Calculate target position changes for each symbol
for symbol in symbols:
    df_backtest[f'{symbol}_target_position_change'] = df_backtest[f'{symbol}_target_position'].diff(1)

# Calculate cash, cumulative cash, position mark, and PNL for each symbol
for symbol in symbols:
    df_backtest[f'{symbol}_cash'] = -df_backtest[f'{symbol}_target_position_change'] * df_backtest[symbol]
    df_backtest[f'{symbol}_cumulative_cash'] = df_backtest[f'{symbol}_cash'].cumsum()
    df_backtest[f'{symbol}_position_mark'] = df_backtest[f'{symbol}_target_position'] * df_backtest[symbol]
    df_backtest[f'{symbol}_pnl'] = df_backtest[f'{symbol}_position_mark'] + df_backtest[f'{symbol}_cumulative_cash']

# Display the PNL for each symbol
for symbol in symbols:
    print(f"{symbol} PNL:")
    display(df_backtest[f'{symbol}_pnl'].tail(3))

In [ ]:
df_backtest

In [ ]:
df_backtest

In [ ]:
df_backtest

In [ ]:
df_backtest

In [ ]:
import plotly.graph_objects as go

# Create traces for ROSES
trace_roses_pred_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_pred_returns'],
    name='ROSES Predicted Returns',
    yaxis='y1'
)

trace_roses_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_returns_in_500_its'],
    name='ROSES Returns',
    yaxis='y3'
)

trace_roses_price = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES'],
    name='ROSES Price',
    yaxis='y4'
)

trace_roses_pnl = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_pnl'],
    name='ROSES PnL',
    yaxis='y2'
)

# Create traces for STRAWBERRIES
trace_strawberries_pred_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES_pred_returns'],
    name='STRAWBERRIES Predicted Returns',
    yaxis='y5'
)

trace_strawberries_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES_returns_in_800_its'],
    name='STRAWBERRIES Returns',
    yaxis='y7'
)

trace_strawberries_price = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES'],
    name='STRAWBERRIES Price',
    yaxis='y8'
)

trace_strawberries_pnl = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES_pnl'],
    name='STRAWBERRIES PnL',
    yaxis='y10'
)

# Create traces for GIFT_BASKET
trace_gift_basket_pred_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['GIFT_BASKET_pred_returns'],
    name='GIFT_BASKET Predicted Returns',
    yaxis='y9'
)

trace_gift_basket_returns = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['GIFT_BASKET_returns_in_900_its'],
    name='GIFT_BASKET Returns',
    yaxis='y11'
)

trace_gift_basket_price = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['GIFT_BASKET'],
    name='GIFT_BASKET Price',
    yaxis='y12'
)

trace_gift_basket_pnl = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['GIFT_BASKET_pnl'],
    name='GIFT_BASKET PnL',
    yaxis='y13'
)

# Create trace for STARFRUIT
trace_starfruit = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STARFRUIT'],
    name='STARFRUIT',
    yaxis='y6'
)

# Create the layout for the graph
layout = go.Layout(
    title='Multiple Symbols - Predicted Returns, Returns, Prices, and PnL',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(title='ROSES Predicted Returns', side='left'),
    yaxis2=dict(title='ROSES PnL', side='right', overlaying='y'),
    yaxis3=dict(title='ROSES Returns', side='right', overlaying='y', position=0.95),
    yaxis4=dict(title='ROSES Price', side='right', overlaying='y', position=0.95),
    yaxis5=dict(title='STRAWBERRIES Predicted Returns', side='right', overlaying='y', position=0.95),
    yaxis6=dict(title='STARFRUIT', side='right', overlaying='y', position=0.95),
    yaxis7=dict(title='STRAWBERRIES Returns', side='right', overlaying='y', position=0.95),
    yaxis8=dict(title='STRAWBERRIES Price', side='right', overlaying='y', position=0.95),
    yaxis9=dict(title='GIFT_BASKET Predicted Returns', side='right', overlaying='y', position=0.95),
    yaxis10=dict(title='STRAWBERRIES PnL', side='right', overlaying='y', position=0.95),
    yaxis11=dict(title='GIFT_BASKET Returns', side='right', overlaying='y', position=0.95),
    yaxis12=dict(title='GIFT_BASKET Price', side='right', overlaying='y', position=0.95),
    yaxis13=dict(title='GIFT_BASKET PnL', side='right', overlaying='y', position=0.95)
)

# Create the figure and add the traces and layout
fig = go.Figure(data=[
    trace_roses_pred_returns, trace_roses_returns, trace_roses_price, trace_roses_pnl,
    trace_strawberries_pred_returns, trace_strawberries_returns, trace_strawberries_price, trace_strawberries_pnl,
    trace_gift_basket_pred_returns, trace_gift_basket_returns, trace_gift_basket_price, trace_gift_basket_pnl,
    trace_starfruit
], layout=layout)

# Show the plot
fig.show()

# backtest day 0 (out of distribution)

In [ ]:
file_name = f"../round1/round-{1}-island-data-bottle/prices_round_{1}_day_0.csv"
df_round_1 = pd.read_csv(file_name, sep=';')

file_name = f"../round3/round-{3}-island-data-bottle/prices_round_{3}_day_0.csv"
df_round_3 = pd.read_csv(file_name, sep=';')

import numpy as np

def market_maker_mid(row):
    if row['product'] == 'STARFRUIT':
        bid_prices = [row['bid_price_1'], row['bid_price_2'], row['bid_price_3']]
        bid_volumes = [row['bid_volume_1'], row['bid_volume_2'], row['bid_volume_3']]
        ask_prices = [row['ask_price_1'], row['ask_price_2'], row['ask_price_3']]
        ask_volumes = [row['ask_volume_1'], row['ask_volume_2'], row['ask_volume_3']]

        best_bid = np.nan
        best_ask = np.nan

        for bid_price, bid_volume in zip(bid_prices, bid_volumes):
            if bid_volume > 15:
                best_bid = bid_price
                break

        for ask_price, ask_volume in zip(ask_prices, ask_volumes):
            if ask_volume > 15:
                best_ask = ask_price
                break

        if not np.isnan(best_bid) and not np.isnan(best_ask):
            return (best_bid + best_ask) / 2
        else:
            return np.nan
    else:
        return row['mid_price']

df_round_1['mid_price'] = df_round_1.apply(market_maker_mid, axis=1)

df_round_1 = df_round_1.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

df_round_3 = df_round_3.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

df = df_round_1.merge(df_round_3, on='timestamp', how='inner')

In [ ]:
file_name = f"../round3/round-{3}-island-data-bottle/prices_round_{3}_day_0.csv"
df_round_3 = pd.read_csv(file_name, sep=';')

In [ ]:
import numpy as np

def market_maker_mid(row):
    if row['product'] == 'STARFRUIT':
        bid_prices = [row['bid_price_1'], row['bid_price_2'], row['bid_price_3']]
        bid_volumes = [row['bid_volume_1'], row['bid_volume_2'], row['bid_volume_3']]
        ask_prices = [row['ask_price_1'], row['ask_price_2'], row['ask_price_3']]
        ask_volumes = [row['ask_volume_1'], row['ask_volume_2'], row['ask_volume_3']]

        best_bid = np.nan
        best_ask = np.nan

        for bid_price, bid_volume in zip(bid_prices, bid_volumes):
            if bid_volume > 15:
                best_bid = bid_price
                break

        for ask_price, ask_volume in zip(ask_prices, ask_volumes):
            if ask_volume > 15:
                best_ask = ask_price
                break

        if not np.isnan(best_bid) and not np.isnan(best_ask):
            return (best_bid + best_ask) / 2
        else:
            return np.nan
    else:
        return row['mid_price']

df_round_1['mid_price'] = df_round_1.apply(market_maker_mid, axis=1)

In [ ]:
df_round_1 = df_round_1.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

In [ ]:
df_round_3 = df_round_3.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

In [ ]:
df = df_round_1.merge(df_round_3, on='timestamp', how='inner')

In [ ]:
df

In [ ]:
df_backtest = df.copy()[['timestamp', 'STARFRUIT', 'ROSES', 'STRAWBERRIES']]

df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 500)
df_backtest = get_prev_returns(df_backtest, 'STRAWBERRIES', 500)
df_backtest = get_future_returns(df_backtest, 'ROSES', 500)



STARFRUIT_BETA = 1.25

df_backtest['ROSES_pred_returns_in_500_its'] = STARFRUIT_BETA * df_backtest['STARFRUIT_returns_from_500_its_ago']

take_threshold = 0.004
clear_threshold = 0.0005


df_backtest['roses_signal'] = df_backtest.apply(lambda row: "SHORT" if row['ROSES_pred_returns_in_500_its'] <= -take_threshold else "CLEAR" if (row['ROSES_pred_returns_in_500_its'] > -clear_threshold and row['ROSES_pred_returns_in_500_its'] < clear_threshold) else "LONG" if row['ROSES_pred_returns_in_500_its'] >= take_threshold else None,  axis=1)

df_backtest['last_roses_signal'] = df_backtest['roses_signal'].fillna(method='ffill')

df_backtest

POSITION = 60

df_backtest.loc[df_backtest['last_roses_signal'] == 'CLEAR', 'roses_target_position'] = 0
df_backtest.loc[df_backtest['last_roses_signal'] == 'SHORT', 'roses_target_position'] = -POSITION
df_backtest.loc[df_backtest['last_roses_signal'] == 'LONG', 'roses_target_position'] = POSITION

df_backtest

df_backtest['roses_target_position'] = df_backtest['roses_target_position'].fillna(0)

df_backtest

df_backtest['roses_target_position_change'] = df_backtest['roses_target_position'].diff(1)

df_backtest = df_backtest.dropna()

df_backtest['cash'] = -df_backtest['roses_target_position_change'] * df_backtest['ROSES']

df_backtest['cumulative_cash'] = df_backtest['cash'].cumsum()

df_backtest['position_mark'] = df_backtest['roses_target_position'] * df_backtest['ROSES']

df_backtest['pnl'] = df_backtest['position_mark'] + df_backtest['cumulative_cash']

df_backtest['pnl'].tail(3)

In [ ]:
df_backtest

In [ ]:
import plotly.graph_objects as go

# Create the first trace for ROSES_pred_returns_in_500_its
trace1 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_pred_returns_in_500_its'],
    name='ROSES Predicted Returns',
    yaxis='y1'
)

# Create the second trace for pnl
trace2 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['pnl'],
    name='PnL',
    yaxis='y2'
)

# Create the third trace for ROSES
trace3 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_returns_in_500_its'],
    name='ROSES_returns',
    yaxis='y3'
)

trace4 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES'],
    name='ROSES',
    yaxis='y4'
)

trace5 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STARFRUIT'],
    name='STARFRUIT',
    yaxis='y5'
)

trace6 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES'],
    name='STRAWBERRIES',
    yaxis='y6'
)



# Create the layout for the graph
layout = go.Layout(
    title='ROSES, Predicted Returns, and PnL',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(
        title='ROSES Predicted Returns',
        side='left'
    ),
    yaxis2=dict(
        title='PnL',
        side='right',
        overlaying='y'
    ),
    yaxis3=dict(
        title='ROSES_returns',
        side='right',
        overlaying='y',
        position=0.95
    ),
    yaxis4=dict(
        title='ROSES',
        side='right',
        overlaying='y',
        position=0.92
    ),
    yaxis5=dict(
        title='STARFRUIT',
        side='right',
        overlaying='y',
        position=0.9
    ),
    yaxis6=dict(
        title='STRAWBERRIES',
        side='right',
        overlaying='y',
        position=0.9
    )
)

# Create the figure and add the traces and layout
fig = go.Figure(data=[trace1, trace2, trace3, trace4, trace5,trace6], layout=layout)

# Show the plot
fig.show()

In [ ]:
import plotly.graph_objects as go

# Create the first trace for ROSES_pred_returns_in_500_its
trace1 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_pred_returns_in_400_its'],
    name='ROSES Predicted Returns',
    yaxis='y1'
)

# Create the second trace for pnl
trace2 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['pnl'],
    name='PnL',
    yaxis='y2'
)

# Create the third trace for ROSES
trace3 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES_returns_in_400_its'],
    name='ROSES_returns',
    yaxis='y3'
)

trace4 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ROSES'],
    name='ROSES',
    yaxis='y4'
)

trace5 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STARFRUIT'],
    name='STARFRUIT',
    yaxis='y5'
)

trace6 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['STRAWBERRIES'],
    name='STRAWBERRIES',
    yaxis='y6'
)



# Create the layout for the graph
layout = go.Layout(
    title='ROSES, Predicted Returns, and PnL',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(
        title='ROSES Predicted Returns',
        side='left'
    ),
    yaxis2=dict(
        title='PnL',
        side='right',
        overlaying='y'
    ),
    yaxis3=dict(
        title='ROSES_returns',
        side='right',
        overlaying='y',
        position=0.95
    ),
    yaxis4=dict(
        title='ROSES',
        side='right',
        overlaying='y',
        position=0.92
    ),
    yaxis5=dict(
        title='STARFRUIT',
        side='right',
        overlaying='y',
        position=0.9
    ),
    yaxis6=dict(
        title='STRAWBERRIES',
        side='right',
        overlaying='y',
        position=0.9
    )
)

# Create the figure and add the traces and layout
fig = go.Figure(data=[trace1, trace2, trace3, trace4, trace5,trace6], layout=layout)

# Show the plot
fig.show()

In [ ]:
df_backtest.to_csv('backtest_roses.csv')

# generalized

In [ ]:
file_name = f"../round1/round-{1}-island-data-bottle/prices_round_{1}_day_0.csv"
df_round_1 = pd.read_csv(file_name, sep=';')

file_name = f"../round3/round-{3}-island-data-bottle/prices_round_{3}_day_0.csv"
df_round_3 = pd.read_csv(file_name, sep=';')

import numpy as np

def market_maker_mid(row):
    if row['product'] == 'STARFRUIT':
        bid_prices = [row['bid_price_1'], row['bid_price_2'], row['bid_price_3']]
        bid_volumes = [row['bid_volume_1'], row['bid_volume_2'], row['bid_volume_3']]
        ask_prices = [row['ask_price_1'], row['ask_price_2'], row['ask_price_3']]
        ask_volumes = [row['ask_volume_1'], row['ask_volume_2'], row['ask_volume_3']]

        best_bid = np.nan
        best_ask = np.nan

        for bid_price, bid_volume in zip(bid_prices, bid_volumes):
            if bid_volume > 15:
                best_bid = bid_price
                break

        for ask_price, ask_volume in zip(ask_prices, ask_volumes):
            if ask_volume > 15:
                best_ask = ask_price
                break

        if not np.isnan(best_bid) and not np.isnan(best_ask):
            return (best_bid + best_ask) / 2
        else:
            return np.nan
    else:
        return row['mid_price']

df_round_1['mid_price'] = df_round_1.apply(market_maker_mid, axis=1)

df_round_1 = df_round_1.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

df_round_3 = df_round_3.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

df = df_round_1.merge(df_round_3, on='timestamp', how='inner')

In [ ]:
from sklearn.metrics import r2_score
def backtest_strategy(df, responder, responder_timeframe, predictors, predictor_betas, predictor_timeframes, take_threshold, clear_threshold, positions):
    df_backtest = df.copy()[[responder] + predictors + ['timestamp']]
    
    # Calculate predictor returns
    for i, predictor in enumerate(predictors):
        df_backtest = get_prev_returns(df_backtest, predictor, predictor_timeframes[i])
    
    # Calculate responder future returns
    df_backtest = get_future_returns(df_backtest, responder, responder_timeframe)
    
    # Calculate predicted returns for the responder
    df_backtest[f'{responder}_pred_returns'] = 0
    for i, predictor in enumerate(predictors):
        df_backtest[f'{responder}_pred_returns'] += predictor_betas[i] * df_backtest[f'{predictor}_returns_from_{predictor_timeframes[i]}_its_ago']
        
    
    df_backtest_copy = df_backtest.copy().dropna(subset=[f'{responder}_pred_returns', f'{responder}_returns_in_{responder_timeframe}_its'])
    # Calculate R-squared between predicted and actual future returns
    r2 = r2_score(df_backtest_copy[f'{responder}_returns_in_{responder_timeframe}_its'], df_backtest_copy[f'{responder}_pred_returns'])
    print(f"R-squared for {responder} predicted returns: {r2:.4f}")

    
    # Generate signals based on predicted returns and thresholds
    def generate_signals(row):
        pred_returns = row[f'{responder}_pred_returns']
        if pred_returns <= -take_threshold:
            return 'SHORT'
        elif -clear_threshold < pred_returns < clear_threshold:
            return 'CLEAR'
        elif pred_returns >= take_threshold:
            return 'LONG'
        else:
            return None
    
    df_backtest[f'{responder}_signal'] = df_backtest.apply(generate_signals, axis=1)
    df_backtest[f'last_{responder}_signal'] = df_backtest[f'{responder}_signal'].fillna(method='ffill')
    
    # Calculate target positions
    df_backtest.loc[df_backtest[f'last_{responder}_signal'] == 'CLEAR', f'{responder}_target_position'] = 0
    df_backtest.loc[df_backtest[f'last_{responder}_signal'] == 'SHORT', f'{responder}_target_position'] = -positions
    df_backtest.loc[df_backtest[f'last_{responder}_signal'] == 'LONG', f'{responder}_target_position'] = positions
    df_backtest[f'{responder}_target_position'] = df_backtest[f'{responder}_target_position'].fillna(0)
    
    # Calculate target position changes
    df_backtest[f'{responder}_target_position_change'] = df_backtest[f'{responder}_target_position'].diff(1)
    
    # Calculate cash, cumulative cash, position mark, and PNL
    df_backtest[f'{responder}_cash'] = -df_backtest[f'{responder}_target_position_change'] * df_backtest[responder]
    df_backtest[f'{responder}_cumulative_cash'] = df_backtest[f'{responder}_cash'].cumsum()
    df_backtest[f'{responder}_position_mark'] = df_backtest[f'{responder}_target_position'] * df_backtest[responder]
    df_backtest[f'{responder}_pnl'] = df_backtest[f'{responder}_position_mark'] + df_backtest[f'{responder}_cumulative_cash']
    
    return df_backtest

In [ ]:
responder = 'ROSES'
responder_timeframe = 500
predictors = [ 'STARFRUIT']
predictor_betas = [1.25]
predictor_timeframes = [500]
take_threshold = 0.00125
clear_threshold = 0.0002
positions = 60

df_backtest = backtest_strategy(df, responder, responder_timeframe, predictors, predictor_betas, predictor_timeframes, take_threshold, clear_threshold, positions)

print(f"{responder} PNL:")
display(df_backtest[f'{responder}_pnl'].tail(3))

In [ ]:
responder = 'GIFT_BASKET'
responder_timeframe = 900
predictors = [ 'STARFRUIT']
predictor_betas = [0.33]
predictor_timeframes = [900]
take_threshold = 0.002
clear_threshold = 0.0002
positions = 60

df_backtest = backtest_strategy(df, responder, responder_timeframe, predictors, predictor_betas, predictor_timeframes, take_threshold, clear_threshold, positions)

print(f"{responder} PNL:")
display(df_backtest[f'{responder}_pnl'].tail(3))

In [ ]:
df_backtest

In [ ]:
import plotly.graph_objects as go

# Create the first trace for ROSES_pred_returns_in_500_its
trace1 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest[f'{responder}_pnl'],
    name=f'{responder} pnl',
    yaxis='y1'
)




# Create the layout for the graph
layout = go.Layout(
    title=f'{responder}, PnL',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(
        title=f'{responder} pnl',
        side='left'
    ),

)

# Create the figure and add the traces and layout
fig = go.Figure(data=[trace1], layout=layout)

# Show the plot
fig.show()

In [ ]:
z

In [ ]:
df_backtest['cash'] = -df_backtest['roses_target_position_change'] * df_backtest['ROSES']

df_backtest['cumulative_cash'] = df_backtest['cash'].cumsum()

df_backtest['position_mark'] = df_backtest['roses_target_position'] * df_backtest['ROSES']

df_backtest['pnl'] = df_backtest['position_mark'] + df_backtest['cumulative_cash']

df_backtest['pnl'].tail(3)

In [ ]:
df_backtest = df_backtest.dropna()

In [ ]:
df_backtest['roses_target_position_change'] = df_backtest['roses_target_position'].diff(1)

In [ ]:
df_backtest

In [ ]:
df_backtest['roses_target_position'] = df_backtest['roses_target_position'].fillna(0)

In [ ]:
df_backtest

In [ ]:
df_backtest.loc[df_backtest['last_roses_signal'] == 'CLEAR', 'roses_target_position'] = 0
df_backtest.loc[df_backtest['last_roses_signal'] == 'SHORT', 'roses_target_position'] = -POSITION
df_backtest.loc[df_backtest['last_roses_signal'] == 'LONG', 'roses_target_position'] = POSITION

In [ ]:
POSITION = 60

In [ ]:
df_backtest

In [ ]:
df_backtest['last_roses_signal'] = df_backtest['roses_signal'].fillna(method='ffill')

In [ ]:

df_backtest['roses_signal'] = df_backtest.apply(lambda row: "SHORT" if row['ROSES_pred_returns_in_500_its'] <= -take_threshold else "CLEAR" if (row['ROSES_pred_returns_in_500_its'] > -clear_threshold and row['ROSES_pred_returns_in_500_its'] < clear_threshold) else "LONG" if row['ROSES_pred_returns_in_500_its'] >= take_threshold else None,  axis=1)

In [ ]:
take_threshold = 0.004
clear_threshold = 0.0005

In [ ]:
df_backtest['ROSES_pred_returns_in_500_its'] = STARFRUIT_BETA * df_backtest['STARFRUIT_returns_from_500_its_ago'] + STRAWBERRIES_BETA * df_backtest['STRAWBERRIES_returns_from_500_its_ago']

In [ ]:
STARFRUIT_BETA = 1.25
STRAWBERRIES_BETA = 0

In [ ]:
df_backtest = get_prev_returns(df_backtest, 'STARFRUIT', 500)
df_backtest = get_prev_returns(df_backtest, 'STRAWBERRIES', 500)
df_backtest = get_future_returns(df_backtest, 'ROSES', 500)

In [ ]:
df_backtest['ROSES_pred_returns_in_500_its'].describe()

In [ ]:
df_round_1

In [ ]:
df_backtest['']

In [ ]:
df_backtest['pnl']

In [ ]:
df_backtest

In [ ]:

df_backtest.to_csv('backtest.csv', index=False)